# ATM Channel Centralized FY2025

This notebook computes ATM Channel risk metrics for FY2025.

**Metrics Computed:**
- **1.1** — Total number of unrated or unscored customers
- **1.2** — Tier 1/2 High Risk Customers
- **1.3** — High Risk Customers (excluding Tier 1/2)
- **1.4** — Medium Risk Customers
- **1.5** — Low Risk Customers (including unscored)
- **1.8** — True Sanctions Matches
- **1.9** — Blocked Accounts / Asset Freezes
- **SD2** — PEP (Politically Exposed Persons)
- **SD6** — Customers with TD less than 1 year
- **3.17** — Unusual Transaction Referrals (UTR)
- **3.18** — SARs/STRs
- **3.19** — LCTRs

**Customer Base:** `SELECT DISTINCT acc.customr_num, acc.customr_type` (original logic, no GROUP BY dedup).

**Filters:** Bank=4, aplictn_id IN (ACS, VSA), customr_status=00, effective_date <= 20251031


## Setup: JDBC Connections & Credentials


In [ ]:
# ------------------------------------------------------------------
# JDBC connection strings and authentication credentials
# ------------------------------------------------------------------
czJdbcUrl = "jdbc:sqlserver://p3001-eastus2-asql-2.database.windows.net:1433;database=eda-akora2-aaecz-corporatepoolprd;loginTimeout=10;"
srzJdbcURL = "jdbc:sqlserver://p3001-eastus2-asql-3.database.windows.net:1433;database=eda-akora2-aaed1-srzpoolprd;loginTimeout=10"
azJdbcURL = "jdbc:sqlserver://p3006-eastus2-asql-1.database.windows.net:1433;database=eda-akora-aaaz-CAGAML00BI0001ClusterPRD;loginTimeout=10"
izJdbcUrl = "jdbc:sqlserver://p3004-eastus2-asql-32.database.windows.net:1433;database=eda-akora-aaicz-icz00001poolprd;loginTimeout=10"

jdbcUsername = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_AppID")
jdbcPassword = dbutils.secrets.get(scope="aaaz-base", key="SP_ADB_AAAZ_CAGAML00BI0001_PRD_PWD")

connectionProperties = {
    "AADSecurePrincipalID" : jdbcUsername,
    "AADSecurePrincipalSecret" : jdbcPassword,
    "driver" : "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "authentication" : "ActiveDirectoryServicePrincipal",
    "fetchsize" : "10"
}

## Build Assessment Unit (AU) Universe

Create the ATM Channel AU table for the reporting date **20251031**.

The universe combines:
- **Personal customers** (customr_type=0) joined with `cif_personal_fy25`
- **Non-personal customers** (customr_type=1) joined with `cif_non_personal_fy25`

Uses the original `SELECT DISTINCT` logic — each (customr_num, customr_type) pair is retained.


In [ ]:
%sql
-- ==================================================================
-- Build the Assessment Unit (AU) universe using the original
-- SELECT DISTINCT logic (no GROUP BY dedup).
-- Filters: bank_num=4, aplictn_id in (ACS,VSA), status=00,
--          effective_date <= 20251031
-- ==================================================================
CREATE OR REPLACE TABLE rafy2025_centralized.atm_au_20251031
USING DELTA
AS
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_personal_fy25 pers
  ON acc.customr_num = pers.customr_num
  AND acc.customr_bank_num = pers.customr_bank_num
  AND acc.customr_type = pers.customr_type
WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 0
  AND acc.aplictn_id IN ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND pers.customr_status = '00'
UNION
SELECT DISTINCT acc.customr_num, acc.customr_type
FROM ra_fy_2025.cif_accounts_fy25 acc
JOIN ra_fy_2025.cif_non_personal_fy25 npers
  ON acc.customr_num = npers.customr_num
  AND acc.customr_bank_num = npers.customr_bank_num
  AND acc.customr_type = npers.customr_type
WHERE acc.customr_bank_num = 4
  AND acc.customr_type = 1
  AND acc.aplictn_id IN ('ACS', 'VSA')
  AND SUBSTRING(acc.ifw_effective_date, 1, 8) <= '20251031'
  AND npers.customr_status = '00' 

## Data Preparation: Load AU & Build CIF Keys


In [ ]:
# ------------------------------------------------------------------
# Imports
# ------------------------------------------------------------------
from pyspark.sql.functions import *

In [ ]:
# ------------------------------------------------------------------
# Load the AU universe and derive standardized CIF keys:
#   cust_no     : 9-digit zero-padded customer number
#   cust_type_mn: 'N' for non-personal (type=1), 'P' for personal
# ------------------------------------------------------------------
au = spark.table("rafy2025_centralized.atm_au_20251031")
au_cif = (
    au
    .withColumn('cust_no', lpad(col('customr_num'), 9, '0'))
    .withColumn('cust_type_mn', when(col('customr_type') == '1', 'N').otherwise('P'))
)

In [ ]:
# ------------------------------------------------------------------
# Read the latest customer record from CAEDW via JDBC.
# Uses row_number() partitioned by cust_id to get the most recent
# record (by to_dt desc). This provides cust_intrl_id linkage.
# ------------------------------------------------------------------
cust_query = """(
    SELECT cust_id, cust_no, cust_type_mn
    FROM (
        SELECT *, row_number() OVER (PARTITION BY cust_id ORDER BY to_dt DESC) AS row_num
        FROM caedw.cust
    ) c
    WHERE c.row_num = 1
) t"""
df_cust = spark.read.jdbc(url=czJdbcUrl, table=cust_query, properties=connectionProperties).cache()

In [ ]:
# ------------------------------------------------------------------
# Enrich the AU universe with CAEDW data (cust_id, cust_intrl_id).
# Left join ensures all AU records are retained even if no CAEDW match.
# ------------------------------------------------------------------
au_enriched = (
    au_cif
    .join(df_cust,
          (au_cif.cust_no == df_cust.cust_no) & (au_cif.cust_type_mn == df_cust.cust_type_mn),
          how='left')
    .drop(df_cust.cust_no)
)

## Load Reference Tables

Load scored, rated, and unrated customer tables once.
These are reused across metrics 1.1–1.5.


In [ ]:
# ------------------------------------------------------------------
# Load reference tables used across multiple metrics.
# Loading once here avoids redundant spark.table() calls.
# ------------------------------------------------------------------

# Scored customers (for metric 1.1)
scored_src = spark.table("rafy2025_centralized.CRR_Scorable_Cust_CA")
scored_cif = scored_src.filter(col('v_entity_id').startswith('CIF'))
scored_cif = (
    scored_cif
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)

# Rated customers (for metrics 1.2, 1.3, 1.4, and Low part of 1.5)
rated_src = spark.table('rafy2025_centralized.customer_scorable_rated_ca')
rated_cif = rated_src.filter(col('v_entity_id').startswith('CIF'))
rated_cif = (
    rated_cif
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)

# Join rated with AU once — reused for 1.2, 1.3, 1.4, 1.5
au_rated = rated_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')

# Unrated customers (for unscored part of 1.5)
unrated_src = spark.table('rafy2025_centralized.customer_scorable_unrated_ca')
unrated_cif = unrated_src.filter(col('v_entity_id').startswith('CIF'))
unrated_cif = (
    unrated_cif
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)
au_unrated = unrated_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')

## Metric 1.1 — Total Number of Unrated or Unscored Customers

Identifies AU customers that do **not** appear in the scored/rated customer table.
Uses a `left_anti` join.


In [ ]:
# ------------------------------------------------------------------
# Metric 1.1 — Unrated / Unscored Customers
# Left-anti join: AU customers NOT found in the scored table.
# ------------------------------------------------------------------
print("Scored CIF distinct customers =", scored_cif.count())

m_1_1 = (
    au_enriched
    .join(scored_cif, on=["cust_no", "cust_type_mn"], how="left_anti")
    .dropDuplicates()
)
print("Metric 1.1 (CIF unscored in AU) =", m_1_1.count())

## Metric 1.2 — Tier 1/2 High Risk Customers

Counts distinct customers in the AU whose risk rating is **Tier 1** or **Tier 2**.


In [ ]:
# ------------------------------------------------------------------
# Metric 1.2 — Tier 1/2 High Risk Customers
# Filter au_rated to risk_rating in ('Tier 1', 'Tier 2')
# ------------------------------------------------------------------
m_1_2 = au_rated.filter(col('risk_rating').isin("Tier 1", "Tier 2"))
m_1_2.agg(countDistinct('cust_intrl_id').alias('1.2')).display()

## Metric 1.3 — High Risk Customers (Excluding Tier 1/2)

Counts distinct customers with a **High** risk rating (but not Tier 1/2).


In [ ]:
# ------------------------------------------------------------------
# Metric 1.3 — High Risk Customers (excluding Tier 1/2)
# ------------------------------------------------------------------
m_1_3 = au_rated.filter(col('risk_rating') == "High")
m_1_3.agg(countDistinct('cust_intrl_id').alias('1.3')).display()

## Metric 1.4 — Medium Risk Customers

Counts distinct customers with a **Medium** risk rating.


In [ ]:
# ------------------------------------------------------------------
# Metric 1.4 — Medium Risk Customers
# ------------------------------------------------------------------
m_1_4 = au_rated.filter(col('risk_rating') == "Medium")
m_1_4.agg(countDistinct('cust_intrl_id').alias('1.4')).display()

## Metric 1.5 — Low Risk Customers (Including Unscored)

Combines:
1. **Low-rated** customers from the rated table
2. **Unscored/unrated** customers from the unrated table


In [ ]:
# ------------------------------------------------------------------
# Metric 1.5 — Low Risk + Unscored Customers
# Combines: Low-rated from rated table + all unscored from unrated table
# ------------------------------------------------------------------
m_1_5_low = au_rated.filter(col('risk_rating') == 'Low')
m_1_5 = m_1_5_low.unionByName(au_unrated, allowMissingColumns=True)
m_1_5.agg(countDistinct('cust_intrl_id').alias('1.5')).display()

## Metric 1.8 — True Sanctions Matches

Joins the true sanctions match table with the AU to count distinct Case#.


In [ ]:
# ------------------------------------------------------------------
# Metric 1.8 — True Sanctions Matches
# Load true sanctions, filter to CIF, derive keys, join with AU.
# Count distinct Case# for customers found in the AU.
# ------------------------------------------------------------------
tsm_src = spark.table('ra_adido_2025.true_sanction_match_2025')
tsm_cif = tsm_src.filter(col("Customer#").startswith('CIF'))
tsm_cif = (
    tsm_cif
    .withColumn('cust_no', substring(col('Customer#'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('Customer#'), 4, 1) == '1', 'N').otherwise('P'))
)
display(tsm_cif)

au_tsm = tsm_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_tsm.agg(countDistinct("Case#").alias('1.8')).display()

## Metric 1.9 — Blocked Accounts / Asset Freezes

Query the sanctions blocked property table.


In [ ]:
%sql
-- ------------------------------------------------------------------
-- Metric 1.9 — Blocked Accounts / Asset Freezes
-- Query the sanctions blocked property table directly.
-- ------------------------------------------------------------------
SELECT * FROM ra_adido_2025.customer_sanctioned_blocked_property_2025;

## SD2 — PEP (Politically Exposed Persons)

Joins the PEP list with the AU to identify PEP-flagged customers,
broken down by PEP type.


In [ ]:
# ------------------------------------------------------------------
# SD2 — PEP (Politically Exposed Persons)
# Load exploded PEP list, filter to CIF, derive keys, join with AU.
# Show total count and breakdown by PEP_TYPE.
# ------------------------------------------------------------------
pep_src = spark.table("ra_adido_2025.pep_list_2025_exploded")
pep_cif = pep_src.filter(col('ENTITY').startswith('CIF'))
pep_cif = (
    pep_cif
    .withColumn('cust_no', substring(col('ENTITY'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('ENTITY'), 8, 1) == '1', 'N').otherwise('P'))
)

au_pep = pep_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_pep.agg(countDistinct('cust_intrl_id').alias('SD2')).display()
au_pep.filter(col("PEP_TYPE").isNotNull()).agg(countDistinct('cust_intrl_id').alias('SD2')).display()

sd2_summary = au_pep.groupBy("PEP_TYPE").count().orderBy("count", ascending=False)
sd2_notNull = au_pep.filter(col("PEP_TYPE").isNotNull()).groupBy("PEP_TYPE").count().orderBy("count", ascending=False)
display(sd2_summary)
display(sd2_notNull)

## SD6 — Customers with TD Less Than 1 Year

Counts customers in the AU who have been with TD for less than one year.


In [ ]:
# ------------------------------------------------------------------
# SD6 — Total number of customers with TD less than 1 year
# Load SD6 table, filter to CIF, derive keys, join with AU.
# ------------------------------------------------------------------
sd6_src = spark.table('rafy2025_centralized.cde_sd_6_1yr_fy2025')
sd6_cif = sd6_src.filter(col('v_entity_id').startswith('CIF'))
sd6_cif = (
    sd6_cif
    .withColumn('cust_no', substring(col('v_entity_id'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('v_entity_id'), 8, 1) == '1', 'N').otherwise('P'))
)
display(sd6_cif)

au_sd6 = sd6_cif.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_sd6.agg(countDistinct('cust_intrl_id').alias('sd6')).display()

## Metric 3.17 — Unusual Transaction Referrals (UTR)

Joins UTR customer details with the AU. Reports distinct UTR IDs and customer count.


In [ ]:
# ------------------------------------------------------------------
# Metric 3.17 — Unusual Transaction Referrals (UTR)
# Load UTR cust_details table, join with AU on cust_no/cust_type_mn.
# Report both distinct UTR IDs and customer count.
# ------------------------------------------------------------------
utr_src = spark.table("rafy2025_centralized.td_utr_cde_3_17_2025_cust_details")

au_utr = (
    utr_src
    .join(au_enriched, on=["cust_no", "cust_type_mn"], how="inner")
)
display(au_utr.limit(30))

# Distinct UTR referral IDs
au_utr.agg(countDistinct("utrid").alias("metric_3_17_distinct")).display()
# Total customer rows
au_utr.agg(count("cust_no").alias("metric_3_17")).display()

## Metric 3.18 — SARs/STRs

Joins SAR/STR data with the AU on matched customer ID and type.
Counts distinct `STR_Internal_Unique_ID`.


In [ ]:
# ------------------------------------------------------------------
# Metric 3.18 — SARs/STRs
# Load SAR table, join with AU on matched customer ID and type.
# Count distinct STR_Internal_Unique_ID.
# ------------------------------------------------------------------
sar_src = spark.table("rafy2025_centralized.td_sar_cde_3_18_2025")

au_sar = (
    sar_src.alias("s")
    .join(
        au_enriched.alias("a"),
        (col("s.STR_Admin_Matched_CustomerID") == col("a.cust_no")) &
        (col("s.STR_Admin_Matched_Customer_Type") == col("a.cust_type_mn")),
        "inner"
    )
)

total_sars = au_sar.agg(countDistinct("STR_Internal_Unique_ID").alias("metric 3.18"))
display(total_sars)

## Metric 3.19 — LCTRs (Large Cash Transaction Reports)


In [ ]:
# ------------------------------------------------------------------
# Metric 3.19 — LCTRs (Large Cash Transaction Reports)
# Two sub-sources: TDS Cowen and standard LCTR.
# Filter to CIF, derive keys, join with AU.
# ------------------------------------------------------------------

# 3.19a — Cowen LCTRs
lctr_cowen_src = spark.table('rafy2025_centralized.cde_3_19_lctr_tds_cowen')
lctr_cowen = (
    lctr_cowen_src
    .withColumn('cust_no', substring(col('beneficiaries_clientNumber'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('beneficiaries_clientNumber'), 4, 1) == '1', 'N').otherwise('P'))
)
au_lctr_cowen = lctr_cowen.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_lctr_cowen.agg(countDistinct('input_file_name').alias('3.19 Cowen')).display()

# 3.19b — Standard LCTRs
lctr_src = spark.table('rafy2025_centralized.cde_3_19_lctr')
lctr = (
    lctr_src
    .withColumn('cust_no', substring(col('beneficiaries_clientNumber'), -9, 9))
    .withColumn('cust_type_mn', when(substring(col('beneficiaries_clientNumber'), 4, 1) == '1', 'N').otherwise('P'))
)
au_lctr = lctr.join(au_enriched, on=['cust_no', 'cust_type_mn'], how='inner')
au_lctr.agg(countDistinct('input_file_name').alias('3.19')).display()